# Tahap 09 — Deployment Streamlit

**Mini Project Mobile Price Classification (Kaggle)**  
Dataset: https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification

Notebook ini **menguji model untuk deployment**, menampilkan prediksi dan grafik di Jupyter/GitHub. **Aplikasi web yang sebenarnya** ada di `09_deployment_streamlit.py` dan dijalankan lewat terminal dengan Streamlit.

- Tidak menggunakan `p14_common.py` atau modul proyek tambahan.
- Menggunakan `models/model_terbaik.joblib` hasil Tahap 08, bukan melatih ulang.
- Perlu versi scikit-learn/joblib yang kompatibel dengan lingkungan saat menyimpan model.
- Pada Google Colab, upload file `model_terbaik.joblib` jika diminta.
- Output yang terlihat di GitHub baru ada setelah notebook dijalankan lalu disimpan.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd()
CANDIDATES = [
    CWD / "models" / "model_terbaik.joblib",
    CWD / "P14" / "models" / "model_terbaik.joblib",
    CWD / "model_terbaik.joblib",
]
MODEL_FILE = next((p for p in CANDIDATES if p.is_file()), None)

if MODEL_FILE is None and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload model_terbaik.joblib hasil Tahap 08:")
    uploaded = files.upload()
    if "model_terbaik.joblib" in uploaded:
        MODEL_FILE = Path.cwd() / "model_terbaik.joblib"

if MODEL_FILE is None:
    raise FileNotFoundError(
        "Model Tahap 08 tidak ditemukan. Jalankan 08_pemilihan_model_terbaik.py "
        "agar terbentuk P14/models/model_terbaik.joblib."
    )

print("Model ditemukan:", MODEL_FILE.resolve())
print("Catatan: hanya memuat model lokal yang dibuat dari proyek sendiri.")

## 1. Data masukan dan feature engineering

Aplikasi menerima **20 spesifikasi asli** dan menghitung **5 fitur turunan** dengan rumus yang identik dengan Tahap 08.

In [ ]:
BASE_FEATURES = [
    "battery_power", "blue", "clock_speed", "dual_sim", "fc", "four_g",
    "int_memory", "m_dep", "mobile_wt", "n_cores", "pc", "px_height",
    "px_width", "ram", "sc_h", "sc_w", "talk_time", "three_g",
    "touch_screen", "wifi",
]
NEW_FEATURES = [
    "pixel_area", "screen_area", "battery_per_weight", "camera_total",
    "connectivity_count",
]
ALL_FEATURES = BASE_FEATURES + NEW_FEATURES
CLASS_NAMES = {0: "Low", 1: "Medium", 2: "High", 3: "Very High"}

# Nilai contoh untuk menguji prediksi (bukan data training tambahan).
example = {
    "battery_power": 1450, "blue": 1, "clock_speed": 2.0, "dual_sim": 1,
    "fc": 5, "four_g": 1, "int_memory": 32, "m_dep": 0.6,
    "mobile_wt": 145, "n_cores": 4, "pc": 13, "px_height": 900,
    "px_width": 1440, "ram": 2200, "sc_h": 12, "sc_w": 6,
    "talk_time": 10, "three_g": 1, "touch_screen": 1, "wifi": 1,
}

def prepare_features(specs):
    missing = set(BASE_FEATURES) - set(specs)
    if missing:
        raise ValueError(f"Fitur input belum lengkap: {sorted(missing)}")
    X = pd.DataFrame([{k: specs[k] for k in BASE_FEATURES}])
    for k in BASE_FEATURES:
        X[k] = pd.to_numeric(X[k], errors="raise")
    X["pixel_area"] = X["px_height"] * X["px_width"]
    X["screen_area"] = X["sc_h"] * X["sc_w"]
    X["battery_per_weight"] = X["battery_power"] / X["mobile_wt"].clip(lower=1)
    X["camera_total"] = X["fc"] + X["pc"]
    X["connectivity_count"] = X[["blue", "four_g", "three_g", "wifi"]].sum(
        axis=1, min_count=4
    )
    return X[ALL_FEATURES]

input_for_model = prepare_features(example)
print("Fitur input asli:", len(BASE_FEATURES))
print("Fitur setelah engineering:", input_for_model.shape[1])
display(input_for_model.T.rename(columns={0: "Nilai"}))

## 2. Memuat model dan menjalankan prediksi

Model berasal dari `models/model_terbaik.joblib`. Notebook **tidak** mengklaim hasil uji baru atau melakukan training ulang.

In [ ]:
trained_model = joblib.load(MODEL_FILE)
expected_columns = getattr(trained_model, "feature_names_in_", None)
if expected_columns is not None and list(expected_columns) != ALL_FEATURES:
    raise ValueError("Kolom model berbeda dari skema 25 fitur Tahap 08")

predicted_class = int(trained_model.predict(input_for_model)[0])
if predicted_class not in CLASS_NAMES:
    raise ValueError(f"Kategori hasil prediksi tidak dikenal: {predicted_class}")

print("=== HASIL PREDIKSI TAHAP 09 ===")
print("Kategori:", predicted_class)
print("Nama kategori:", CLASS_NAMES[predicted_class])
print("Catatan: bukan harga dalam rupiah")

display(pd.DataFrame([{
    "predicted_price_range": predicted_class,
    "kategori": CLASS_NAMES[predicted_class],
    "jumlah_fitur": len(ALL_FEATURES),
}]))

## 3. Visualisasi hasil prediksi

Jika algoritma mendukung `predict_proba` (misalnya Random Forest), tampilkan perkiraan probabilitas per kelas. Jika model terpilih SVM tanpa probabilitas, cukup tampilkan label yang diprediksi.

In [ ]:
if hasattr(trained_model, "predict_proba"):
    probabilities = np.asarray(trained_model.predict_proba(input_for_model)[0])
    classes = [int(k) for k in trained_model.classes_]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([f"{k} - {CLASS_NAMES[k]}" for k in classes], probabilities)
    ax.set(title="Estimasi Probabilitas Prediksi per Kelas", ylabel="Probabilitas", ylim=(0, 1.05))
    fig.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar([f"{predicted_class} - {CLASS_NAMES[predicted_class]}"], [1.0])
    ax.set(title="Hasil Klasifikasi (bukan probabilitas)", ylim=(0, 1.1), ylabel="Indikator kelas terpilih")
    ax.text(0, 0.5, "Kelas yang diprediksi", ha="center", va="center", color="white")
    fig.tight_layout()
    plt.show()
    print("Model ini tidak menyediakan predict_proba; grafik hanya menandai kelas hasil prediksi.")

## 4. Menjalankan aplikasi Streamlit

Notebook ini untuk pemeriksaan prediksi dan dokumentasi output. **Untuk membuka aplikasi web dengan form input lengkap**, jalankan perintah berikut pada CMD Windows di folder `P14`:

```cmd
python -m pip install streamlit
python -m streamlit run 09_deployment_streamlit.py
```

Server lokal Streamlit akan menampilkan alamat di terminal, biasanya `http://localhost:8501`. Pastikan `P14/models/model_terbaik.joblib` tetap tersedia. Dataset `train.csv` tidak perlu dibaca saat inferensi.

Setelah notebook berhasil dijalankan, simpan file `.ipynb` untuk menampilkan output-nya di GitHub.